# XD2-Net — 04: Explanations

Extracts and visualises both levels of inherent explainability:

- **Level 1 (temporal):** alpha, i.e. which events mattered. For `hierarchical` it is the
  `[CASE]` self-attention row; see notebook 05 for its per-field (beta) breakdown.
- **Level 2 (feature):** thetas, i.e. which aggregated features mattered.

Also evaluates explanation quality: feature agreement and effective complexity.

**Prerequisite:** run notebooks 01 and 02 first with the same `MODEL` / `DATASET_CONFIG`.

In [ ]:
import os, sys, glob, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab: clone the repo into /content (code) and write outputs to Drive (they survive the runtime).
REPO_URL = 'https://github.com/bemali/xd2-net-pytorch.git'
BRANCH = 'feattransformer-self-attention-exp'       # switch to 'main' once merged
OUTPUT_ROOT = None                                   # None = configs' workspace_dir (./outputs)

if IN_COLAB:
    REPO_ROOT = '/content/xd2-net-pytorch'
    if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
        !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
    %pip install -e {REPO_ROOT} --quiet
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/xd2net_outputs'
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
else:
    REPO_ROOT = os.path.abspath('..')

sys.path.insert(0, REPO_ROOT)   # importable without a kernel restart after pip install -e
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT, '| commit:',
      subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip())
print('OUTPUT_ROOT:', OUTPUT_ROOT)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU runtime)')
except ImportError:
    pass

In [ ]:
# --- parameters --------------------------------------------------------------
MODEL = 'baseline'                                   # 'baseline' | 'hierarchical'
DATASET_CONFIG = 'configs/datasets/bpic2012.yaml'    # any dataset YAML (see configs/datasets/_template.yaml)
MODEL_CONFIG = None                                  # None = configs/models/<MODEL>.yaml
EXPLAIN_BUCKETS = None                               # None = every trained bucket

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression

from xd2net.common.config import load_dataset_config, load_experiment_config
from xd2net.common.data import make_test_loader
from xd2net.common.metrics import evaluate_predictions, feature_agreement
from xd2net.common.device import accelerated, resolve_device
from xd2net.common.registry import get_spec

spec = get_spec(MODEL)
exp = load_experiment_config(MODEL_CONFIG or spec.default_config, spec.model_config_cls)
ds = load_dataset_config(DATASET_CONFIG)
if OUTPUT_ROOT:
    exp.workspace_dir = OUTPUT_ROOT
run_dir = os.path.join(exp.workspace_dir, MODEL, ds.name)
artifacts_dir = os.path.join(run_dir, 'artifacts')
ckpt_dir = os.path.join(run_dir, 'checkpoints')
device = resolve_device(exp.training.device)
data_device = device if (exp.training.data_on_device and accelerated(device)) else None   # GPU-resident batches
exp_dir = os.path.join(run_dir, 'explanations')
os.makedirs(exp_dir, exist_ok=True)

def available_buckets():
    """Bucket keys with a saved artifact, in prefix-length order."""
    keys = [os.path.basename(p)[len('bucket_'):-len('.pkl')]
            for p in glob.glob(os.path.join(artifacts_dir, 'bucket_*.pkl'))]
    return sorted(keys, key=lambda k: int(k.split('_')[1]))

In [ ]:
def load_model_for_bucket(bucket_key):
    """Rebuild the model from its saved config and load the best checkpoint."""
    with open(os.path.join(ckpt_dir, f'model_config_{bucket_key}.json')) as f:
        model = spec.model_cls.from_config(json.load(f))
    checkpoints = glob.glob(os.path.join(ckpt_dir, f'model_{bucket_key}_*.pt'))
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoint for {bucket_key} in {ckpt_dir}')
    best = min(checkpoints, key=lambda p: float(p.split('vl')[-1].replace('.pt', '')))
    model.load_state_dict(torch.load(best, map_location='cpu', weights_only=True))
    print(f'  Loaded: {os.path.basename(best)}')
    return model

In [ ]:
# ── Process selected explanation buckets ──────────────────────────────────────
explain_buckets = EXPLAIN_BUCKETS or available_buckets()
all_thetas = {}  # bucket → (N, exp_dim)
all_feature_names = {}
lr_coefs = {}

for bucket_key in explain_buckets:
    art_path = os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl')
    if not os.path.exists(art_path):
        print(f'{bucket_key}: artifact not found'); continue

    with open(art_path, 'rb') as f:
        art = pickle.load(f)

    model = load_model_for_bucket(bucket_key).to(device)
    test_loader = make_test_loader(art['inputs_test'], art['outputs_test'], art['lengths_test'],
                                   batch_size=exp.training.batch_size, device=data_device)

    explainer = spec.explainer_cls(model, device=exp.training.device)
    df_pred, df_exp = explainer.explain_loader(
        test_loader, art['caseids_test'], art['feature_names'], bucket=bucket_key)

    df_pred.to_csv(os.path.join(exp_dir, f'predictions_{bucket_key}.csv'), index=False)
    df_exp.to_csv(os.path.join(exp_dir, f'explanations_{bucket_key}.csv'), index=False)

    # Store thetas for agreement analysis
    # Re-run inference to get raw thetas
    model.eval()
    thetas_list = []
    with torch.no_grad():
        for batch in test_loader:
            ri, rs, ie, lb, ln = batch
            th, _, _ = model(*(t.to(device) for t in (ri, rs, ie, ln)))
            thetas_list.append(th.cpu().numpy())
    all_thetas[bucket_key] = np.vstack(thetas_list)
    all_feature_names[bucket_key] = art['feature_names']

    # LR baseline coefficients
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(art['inputs_train']['input_exp'], art['outputs_train']['act_output'])
    lr_coefs[bucket_key] = lr.coef_.flatten()

    print(f'{bucket_key}: explanations saved')

In [ ]:
# ── Global feature importance: XD2-Net vs LR ─────────────────────────────────
for bucket_key in explain_buckets:
    if bucket_key not in all_thetas: continue

    thetas = all_thetas[bucket_key]          # (N, F)
    feat_names = all_feature_names[bucket_key]
    coefs = lr_coefs[bucket_key]

    mean_th = thetas.mean(axis=0)
    abs_coef = np.abs(coefs) / (np.abs(coefs).sum() + 1e-9)
    # Align lengths if needed
    f = min(len(mean_th), len(abs_coef), len(feat_names))
    mean_th = mean_th[:f]; abs_coef = abs_coef[:f]; feat_names_f = feat_names[:f]

    top_k = 10
    order = np.argsort(mean_th)[::-1][:top_k]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].barh([feat_names_f[i] for i in order[::-1]], mean_th[order[::-1]])
    axes[0].set_title(f'{bucket_key} — XD2-Net (top {top_k})')
    axes[0].set_xlabel('Mean theta weight')

    lr_order = np.argsort(abs_coef)[::-1][:top_k]
    axes[1].barh([feat_names_f[i] for i in lr_order[::-1]], abs_coef[lr_order[::-1]])
    axes[1].set_title(f'{bucket_key} — Logistic Regression (top {top_k})')
    axes[1].set_xlabel('Normalized |coefficient|')

    plt.tight_layout(); plt.show()

In [ ]:
# ── Feature agreement across buckets ─────────────────────────────────────────
agreement_rows = []
for bucket_key in explain_buckets:
    if bucket_key not in all_thetas: continue
    thetas = all_thetas[bucket_key]
    coefs  = lr_coefs[bucket_key]
    f = min(thetas.shape[1], len(coefs))
    ag = feature_agreement(thetas[:, :f], coefs[:f], top_k=5)
    agreement_rows.append({'bucket': bucket_key, **ag})

df_agreement = pd.DataFrame(agreement_rows)
print(df_agreement.to_string(index=False))

In [ ]:
# ── Effective complexity: iterative feature ablation ─────────────────────────
# For the first explain bucket: remove least important features one by one,
# measure how prediction RMSE degrades.

bucket_key = explain_buckets[0]
if bucket_key in all_thetas:
    art_path = os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl')
    with open(art_path, 'rb') as f:
        art = pickle.load(f)

    model = load_model_for_bucket(bucket_key).to(device)
    explainer = spec.explainer_cls(model, device=exp.training.device)

    ri = torch.tensor(art['inputs_test']['input_dynamic']).to(device)
    rs = torch.tensor(art['inputs_test']['input_static']).to(device)
    ie = torch.tensor(art['inputs_test']['input_exp']).to(device)
    ln = torch.tensor(art['lengths_test']).to(device)

    # Full prediction
    model.eval()
    with torch.no_grad():
        _, full_pred, _ = model(ri, rs, ie, ln)
    full_pred = full_pred.cpu().numpy()

    thetas = all_thetas[bucket_key]  # (N, F)
    mean_th = thetas.mean(axis=0)
    # Order features from least to most important
    removal_order = np.argsort(mean_th)

    rmses = []
    ie_ablated = art['inputs_test']['input_exp'].copy()

    for step, feat_idx in enumerate(removal_order):
        ie_ablated[:, feat_idx] = 0.0
        with torch.no_grad():
            _, ablated_pred, _ = model(ri, rs, torch.tensor(ie_ablated).to(device), ln)
        rmse = float(np.sqrt(np.mean((full_pred - ablated_pred.cpu().numpy())**2)))
        rmses.append(rmse)

    plt.figure(figsize=(8, 4))
    pct_removed = np.linspace(0, 100, len(rmses))
    plt.plot(pct_removed, rmses)
    plt.axhline(0.1, color='r', linestyle='--', label='RMSE = 0.1 threshold')
    plt.xlabel('% features removed (least important first)')
    plt.ylabel('Prediction RMSE vs. full model')
    plt.title(f'Effective Complexity — {bucket_key}')
    plt.legend(); plt.tight_layout(); plt.show()

    # Find the % of features that can be removed before RMSE > 0.1
    effective_pct = next((pct_removed[i] for i, r in enumerate(rmses) if r > 0.1), 100.0)
    print(f'Up to {effective_pct:.1f}% of features can be removed before RMSE > 0.1')

In [ ]:
# ── Temporal attention heatmap for one example case ───────────────────────────
bucket_key = explain_buckets[-1]  # use the longest bucket
if bucket_key in all_thetas:
    art_path = os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl')
    with open(art_path, 'rb') as f:
        art = pickle.load(f)

    model = load_model_for_bucket(bucket_key).to(device)
    explainer = spec.explainer_cls(model, device=exp.training.device)

    # Pick the first 5 test cases
    n_show = 5
    ri = torch.tensor(art['inputs_test']['input_dynamic'][:n_show])
    rs = torch.tensor(art['inputs_test']['input_static'][:n_show])
    ie = torch.tensor(art['inputs_test']['input_exp'][:n_show])
    ln = torch.tensor(art['lengths_test'][:n_show])

    alpha = explainer.get_temporal_attention(ri, rs, ie, ln)  # (5, T)

    fig, ax = plt.subplots(figsize=(12, 3))
    sns.heatmap(alpha, ax=ax, cmap='Blues', cbar_kws={'label': 'Attention weight'})
    ax.set_xlabel('Event position (most recent = position 0)')
    ax.set_ylabel('Case')
    ax.set_title(f'Temporal attention (alpha) — {bucket_key}')
    ax.set_yticklabels([str(art['caseids_test'][i]) for i in range(n_show)], rotation=0)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Save feature agreement results ────────────────────────────────────────────
out_dir = os.path.join(run_dir, 'results')
os.makedirs(out_dir, exist_ok=True)
df_agreement.to_csv(os.path.join(out_dir, 'feature_agreement.csv'), index=False)
print('Saved to', out_dir)